# R&D 3-A: On-Device Deployment — PyTorch → ONNX → TFLite Conversion

**목적**: 모바일 기기에서 실시간 STT 추론을 위해 파인튜닝된 Whisper 모델을 TFLite 포맷으로 변환하는 파이프라인을 구축.

**시도한 내용**:
- PyTorch → ONNX 변환 후 `onnx-tensorflow`로 SavedModel 변환
- `TFLiteConverter`를 통한 `.tflite` 파일 추출
- Serving 시그니처 직접 정의 (하드코딩된 `forced_decoder_ids`, `max_new_tokens`)

**결론**: TFLite 변환에는 성공했으나 Inference 테스트 결과 `[50258, 50264... 918, 918]`과 같이 무의미한 토큰을 반복 출력. 모델 구조 변환 과정의 정보 손실 및 연산자 호환성 문제로 파악. 클라우드 서버 기반 아키텍처로 피벗.


https://github.com/sithu31296/PyTorch-ONNX-TFLite

In [79]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [80]:
!pip install tensorflow==2.15
!pip install tensorflow-probability==0.23

In [81]:
!git clone https://github.com/onnx/onnx-tensorflow.git
%cd onnx-tensorflow
!pip install -e .

Cloning into 'onnx-tensorflow'...
remote: Enumerating objects: 6516, done.
remote: Counting objects: 100% (465/465), done.
remote: Compressing objects: 100% (200/200), done.
remote: Total 6516 (delta 325), reused 383 (delta 261), pack-reused 6051 (from 1)
Receiving objects: 100% (6516/6516), 1.98 MiB | 10.57 MiB/s, done.
Resolving deltas: 100% (5050/5050), done.
/content/onnx-tensorflow/onnx-tensorflow
Obtaining file:///content/onnx-tensorflow/onnx-tensorflow
  Preparing metadata (setup.py) ... done
  Attempting uninstall: onnx-tf
    Found existing installation: onnx-tf 1.10.0
    Uninstalling onnx-tf-1.10.0:
      Successfully uninstalled onnx-tf-1.10.0
  Running setup.py develop for onnx-tf


In [82]:
!pip install onnx
!pip install onnxruntime

In [83]:
_BASE_DIR = '/content/drive/MyDrive/JBNU/2024-2/Capstone/Whisper/'
_DATASETS_DIR = _BASE_DIR + 'datasets/'
_TRANSCRIPT_DIR = _DATASETS_DIR + 'transcript/'

In [84]:
onnx_model_path = _BASE_DIR + 'whisper-infant.onnx'
tf_model_path = _BASE_DIR + 'tensorflow/'
tflite_model_path = _BASE_DIR + 'tensorflow/whisper-infant.tflite'

In [85]:
model_name = _BASE_DIR + 'Telos_LLM'
tf_model_name = _BASE_DIR + 'tensorflow'
processor_name = _BASE_DIR + 'Telos_LLM_Processor'

In [8]:
import onnx

# ONNX 모델 로드
onnx_model = onnx.load(_BASE_DIR+"whisper-infant.onnx")

# 모델 검증
try:
    onnx.checker.check_model(onnx_model)
    print("ONNX 모델이 유효합니다.")
except onnx.checker.ValidationError as e:
    print("ONNX 모델 검증 실패:", e)


ONNX 모델이 유효합니다.


In [9]:
import onnxruntime as ort

# ONNX Runtime 세션 생성
ort_session = ort.InferenceSession(_BASE_DIR+"whisper-infant.onnx")

# 입력 정보 확인
input_names = [input.name for input in ort_session.get_inputs()]
print("모델 입력 이름:", input_names)

# 출력 정보 확인
output_names = [output.name for output in ort_session.get_outputs()]
print("모델 출력 이름:", output_names)

모델 입력 이름: ['input_features', 'decoder_input_ids']
모델 출력 이름: ['logits', 'onnx::Shape_775', 'onnx::MatMul_780', '876', 'onnx::MatMul_881', 'onnx::Shape_990', 'onnx::MatMul_995', '1091', 'onnx::MatMul_1096', 'onnx::Shape_1205', 'onnx::MatMul_1210', '1306', 'onnx::MatMul_1311', 'onnx::Shape_1420', 'onnx::MatMul_1425', '1521', 'onnx::MatMul_1526', 'onnx::MatMul_657']


In [86]:
!pip install datasets

In [87]:
from datasets import Dataset, Audio, DatasetDict, load_from_disk
import pandas as pd
import os

names = ['train', 'test']
dataset = DatasetDict()

for name in names:
    df = pd.read_csv(_TRANSCRIPT_DIR + name + '.csv')
    df['audio'] = df['file'].apply(lambda x: os.path.join(_DATASETS_DIR, str(x)))

    dataset[name] = Dataset.from_pandas(df)
    dataset[name] = dataset[name].cast_column('audio', Audio(sampling_rate=16000))

In [114]:
import torch
from transformers import WhisperForConditionalGeneration, WhisperProcessor
import numpy as np

model = WhisperForConditionalGeneration.from_pretrained(model_name)
processor = WhisperProcessor.from_pretrained(processor_name)

model.eval()

# 모델을 CPU로 이동
device = torch.device('cpu')
model.to(device)

inputs = processor.feature_extractor(
    dataset['test'][1]["audio"]["array"], sampling_rate=dataset['test'][1]["audio"]["sampling_rate"], return_tensors="pt"
)
input_features = inputs.input_features


# Dummy decoder input IDs (batch_size=1, sequence_length=1)
decoder_input_ids = torch.tensor([[model.config.decoder_start_token_id]], device=device)

In [90]:
forced_decoder_ids

[(1, 50264), (2, 50359), (3, 50363)]

In [116]:
decoder_input_ids = torch.tensor([[50264]])

In [111]:
input_features.shape

(1, 80, 3000)

In [117]:
# 입력 데이터 설정
ort_inputs = {
    "input_features": np.array(input_features),
    "decoder_input_ids": np.array(decoder_input_ids),
}

# 추론 실행
ort_outs = ort_session.run(None, ort_inputs)

# 출력 얻기
logits = ort_outs[0]
print("추론 결과 logits 크기:", logits.shape)
logits

추론 결과 logits 크기: (1, 1, 51865)


array([[[-25.416618, -28.241549, -23.068308, ..., -25.396961,
         -26.707   , -25.889084]]], dtype=float32)

In [130]:
import numpy as np

# Logits shape: [batch_size, seq_length, vocab_size]
# Example logits output from your model:
# logits = np.array([...])

# Convert logits to token IDs by taking the argmax along the last dimension
predicted_token_ids = np.argmax(logits, axis=-1)

print("Predicted token IDs:", predicted_token_ids)

# Decode the token IDs
transcription = processor.tokenizer.decode(predicted_token_ids[0], skip_special_tokens=True)

print("Transcription:", transcription)

Predicted token IDs: [[50359]]
Transcription: 


In [118]:
import torch
from transformers import WhisperForConditionalGeneration, WhisperProcessor
import numpy as np

model = WhisperForConditionalGeneration.from_pretrained(model_name)
processor = WhisperProcessor.from_pretrained(processor_name)

model.eval()

# 모델을 CPU로 이동
device = torch.device('cpu')
model.to(device)


inputs = processor.feature_extractor(
    dataset['test'][1]["audio"]["array"], sampling_rate=dataset['test'][1]["audio"]["sampling_rate"], return_tensors="pt"
)
input_features = inputs.input_features


# Dummy decoder input IDs (batch_size=1, sequence_length=1)
decoder_input_ids = torch.tensor([[50264]])#torch.tensor([[model.config.decoder_start_token_id]], device=device)

# 추론 수행하여 logits 얻기
with torch.no_grad():
    outputs = model(input_features=input_features, decoder_input_ids=decoder_input_ids)
    logits_pt = outputs.logits  # logits 텐서


logits_pt

tensor([[[-25.4167, -28.2416, -23.0684,  ..., -25.3970, -26.7070, -25.8891]]])

In [119]:
difference = torch.abs(torch.from_numpy(logits) - logits_pt)

# Compute the maximum absolute difference
max_difference = torch.max(difference)
print("Maximum absolute difference:", max_difference.item())

# Compute the mean absolute difference
mean_difference = torch.mean(difference)
print("Mean absolute difference:", mean_difference.item())

# Compute the mean squared error (MSE)
mse = torch.mean((torch.from_numpy(logits) - logits_pt) ** 2)
print("Mean squared error:", mse.item())

Maximum absolute difference: 8.0108642578125e-05
Mean absolute difference: 4.9541471526026726e-05
Mean squared error: 2.4850874691395575e-09


In [96]:
import onnx
from onnx_tf.backend import prepare

onnx_model = onnx.load(onnx_model_path)

tf_rep = prepare(onnx_model)

tf_rep.export_graph(tf_model_path)

In [97]:
import tensorflow as tf

converter = tf.lite.TFLiteConverter.from_saved_model(tf_model_path)
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,       # TFLite built-in ops
    tf.lite.OpsSet.SELECT_TF_OPS          # Select TensorFlow ops
]
tflite_model = converter.convert()

# Save the model
with open(tflite_model_path, 'wb') as f:
    f.write(tflite_model)


In [98]:
import tensorflow as tf

# TFLite 모델 로드 및 텐서 할당
interpreter = tf.lite.Interpreter(model_path=tflite_model_path)
interpreter.allocate_tensors()


In [99]:
# 입력 및 출력 텐서 정보 가져오기
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("입력 정보:", input_details)
print("출력 정보:", output_details)

입력 정보: [{'name': 'serving_default_decoder_input_ids:0', 'index': 0, 'shape': array([1, 1], dtype=int32), 'shape_signature': array([ 1, -1], dtype=int32), 'dtype': <class 'numpy.int64'>, 'quantization': (0.0, 0), 'quantization_parameters': {'scales': array([], dtype=float32), 'zero_points': array([], dtype=int32), 'quantized_dimension': 0}, 'sparsity_parameters': {}}, {'name': 'serving_default_input_features:0', 'index': 1, 'shape': array([ 1, 80,  1], dtype=int32), 'shape_signature': array([ 1, 80, -1], dtype=int32), 'dtype': <class 'numpy.float32'>, 'quantization': (0.0, 0), 'quantization_parameters': {'scales': array([], dtype=float32), 'zero_points': array([], dtype=int32), 'quantized_dimension': 0}, 'sparsity_parameters': {}}]
출력 정보: [{'name': 'StatefulPartitionedCall:12', 'index': 1146, 'shape': array([1, 1, 1, 1], dtype=int32), 'shape_signature': array([-1, -1, -1, -1], dtype=int32), 'dtype': <class 'numpy.float32'>, 'quantization': (0.0, 0), 'quantization_parameters': {'scales':

In [121]:
import numpy as np
import librosa

inputs = processor.feature_extractor(
    dataset['test'][1]["audio"]["array"],
    sampling_rate=dataset['test'][1]["audio"]["sampling_rate"],
    return_tensors="np"  # Return NumPy array
)

# Get `input_features` and ensure it's of type float32
input_features = inputs.input_features.astype(np.float32)

# Create `decoder_input_ids` as an int64 NumPy array
start_token_id = processor.tokenizer.pad_token_id  # Or use the appropriate start token ID
decoder_input_ids = np.array([[50264]])#np.array([[start_token_id]], dtype=np.int64)


In [122]:
input_features_reduced = np.mean(input_features, axis=2, keepdims=True)
input_features_reduced.shape

(1, 80, 1)

In [123]:
# Set the input tensors

interpreter.set_tensor(input_details[0]['index'], decoder_input_ids)
interpreter.set_tensor(input_details[1]['index'], input_features_reduced)

In [124]:
# 추론 실행
interpreter.invoke()

# 출력 텐서 가져오기
output_data = interpreter.get_tensor(output_details[0]['index'])

output_data.shape

(1, 6, 1500, 64)

In [126]:
# 로짓에서 토큰 IDs 추출
predicted_ids = np.argmax(output_data, axis=-1)

# Check the type and content
print("Type of predicted_ids[0]:", type(predicted_ids[0]))
print("predicted_ids[0]:", predicted_ids[0])

# Convert predicted_ids[0] to a list of integers
token_ids = predicted_ids[0].tolist()

# 토큰 IDs를 텍스트로 디코딩
transcription = processor.tokenizer.decode(token_ids[0], skip_special_tokens=True)
print("전사 결과:", transcription)


Type of predicted_ids[0]: <class 'numpy.ndarray'>
predicted_ids[0]: [[62 23 23 ... 42 23 58]
 [46 61 38 ... 61 38  0]
 [26 34 34 ... 42 53  3]
 [37 38 29 ... 38 38  9]
 [48 51 51 ... 44 44 48]
 [48 29 29 ... 56 48 11]]
전사 결과: _88888888Z88ZZZZZCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCC888CCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCC8888888888888888888888CCCCCCCC8888888888CCCCCCCCCCCCCCCCCC88888CCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCC